# 03a · GasWindTunnel: R3 encoder features

**Purpose.** Inside each fold, train the shared 1D CNN encoder + a temporary linear layer on the training labels only, drop the linear layer, freeze the encoder and extract features. Fit the 4 heads on both tasks.

**Inputs**
- GasWindTunnel `X_signals.npy` (600, 72, 1500) + `meta.csv`

**Outputs**
- Per sensor: (1, 1500) → (16, 750) → (32, 375) → GAP (32,)
- F_enc: (600, 72, 32) → (600, 2304)
- `results/03a_gwt_encoder.csv`

In [ ]:
import sys
sys.path.append("../..")          # project root, so that `import src` works

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("../..")
from src.cv import TASKS, load_baseline_level, load_processed, precomputed, run_cv, save_results, summarize
from src.features.encoder import encoder_builder
# about 30 encoder trainings (5 folds x 3 seeds x 2 tasks): this notebook is the slow one
DATASET = "gwt"
REP = "R3_encoder"
NAME = "03a_gwt_encoder"          # results/03a_gwt_encoder.csv

X, meta, info = load_processed(DATASET)    # X: (N, C, T) baseline corrected
print(X.shape, meta["y_multi"].value_counts().sort_index().tolist())

## R3: shared 1D CNN encoder, 32 features per sensor
The encoder is trained **inside each fold** on the training fold only (encoder + temporary linear layer,
early stopping on an inner validation split). The linear layer is then dropped, the encoder is frozen and
encodes the training and test curves; the 4 heads are fitted on those features exactly as for R1 and R2.

Shape check on 2 samples (untrained network):

In [ ]:
import torch
from src.features.encoder import SensorEncoder

with torch.no_grad():
    z = SensorEncoder(32)(torch.tensor(X[:2]))                     # (2, C * 32)
print(z.shape, "-> full feature matrix", (len(X), z.shape[1]))
assert (len(X), z.shape[1]) == (600, 2304)

## Cross validation: 4 heads × 5 folds × 3 seeds, both tasks
Everything that is fitted (scaler, heads, hyperparameters, encoder) sees the training fold only.

In [ ]:
scores, preds = [], []
for task in TASKS:                                   # "multi", then "binary"
    s, p = run_cv(X, meta, DATASET, task, REP, encoder_builder(n_out=32))
    scores.append(s)
    preds.append(p)
scores, preds = pd.concat(scores, ignore_index=True), pd.concat(preds, ignore_index=True)
save_results(scores, preds, NAME)
print("saved results/" + NAME + ".csv")

## Results: mean ± std over 3 seeds × 5 folds

In [ ]:
print("macro F1"); display(summarize(scores))
print("balanced accuracy"); display(summarize(scores, "balanced_acc"))

In [ ]:
print("ROC AUC, binary task"); summarize(scores[scores["task"] == "binary"], "roc_auc")